<div style="text-align: center;">
    <h1>RAG SYSTEM FOR FINANCIAL ANALYSIS:</h1>
</div>

# 0. PRELIMINARY

## LIBRARIES

In [ ]:
# LIBRARIES

# chunking and reading pdf
!pip install pymupdf langchain langchain-text-splitters -q
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter

# embeding and indexing
!pip install sentence-transformers rank-bm25 faiss-cpu -q
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import numpy as np

# reranking
from sentence_transformers import CrossEncoder

# llm setup
!pip install langchain-groq -q
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from kaggle_secrets import UserSecretsClient

# gradio
!pip install gradio -q
import gradio as gr

# 1. CHUNKING

In [ ]:
# CHUNKING
# devide the PDF into chunks of fixed size (fixed number of characters)
# thsi way we will be able to retrieve and give the LLM only the most relevant chunks instead of the whole PDF

# define the chunking function
# it receives as input the PDF, the chunk size and the overlap
# we pick chunk size 2000 characters (around 500 tokens) because 10-K documents contain long sections 
# so 2000 is large enough to preserve context and small enough to keep the retrieval precise and not dilute chunks
# 400 overlap (20% of chunk size) is standard practice
def chunking(pdf_path, chunk_size=2000, chunk_overlap=400):
     
    # open the PDF file from the Kaggle path where it is saved
    doc = fitz.open(pdf_path)
    
    # first we need to extract the text of all the pages and put it all together in a single string
    # this way we can then chunk that string
    full_document = "\n".join([page.get_text() for page in doc])

    # then initialize the splitter that will split the string with the selected chunk size and overlap
    # separator means that we do not split words but we split at the closest paragraph/newline/space in this priority order
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    # now do the splitting
    chunks = splitter.create_documents([full_document])

    # return the produced chunks
    return chunks

# 2. EMBEDDING AND INDEXING

In [ ]:
# EMBEDDING AND INDEXING
# we embed the chunks and create the faiss and bm25 indexes to search both for meaning and keywords

# import the model that will be used to embed both chunks and queries
# we chose this model because it is much faster than all-mpnet-base-v2 and the quality loss is negligible
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# this function receives the chunks, embeds them and craetes the two indexes
def embedding_indexing(chunks):

    
    # EMBEDDING:
    # first thinkg first we embed the chunks
    
    # extract the raw text (corpus) from the chunks
    corpus = [chunk.page_content for chunk in chunks]

    # then embed the chunks
    embeddings = embedding_model.encode(
        corpus,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    
    # FAISS INDEX (VECTOR DB):
    # the first index is faiss that will allow us to look for relevant chunks based on cosine similarity/meaning
    
    # first normalize the embeddings by dividing them by their norm
    # this way every vector becomes length 1 so the inner product of two embeddings will be their cosine similarity
    faiss.normalize_L2(embeddings)
    
    # then create the faiss index with all the chunks
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)
    faiss_index.add(embeddings)


    # BM25 INDEX:
    # the second index is bm25 that will allow us to look for relevant chunks based on keywords
    
    # tokenize the chunks
    tokenized_corpus = [chunk.lower().split() for chunk in corpus]
    
    # build bm25 index
    bm25_index = BM25Okapi(tokenized_corpus)


    return corpus, faiss_index, bm25_index


# 3. HYBRID RETRIEVAL

In [ ]:
# HYBRID RETRIEVAL
# the following function takes a query and returns the top n most relevant chunks
# it runs two independent searches: semantic (FAISS) and keyword (BM25)
# then combines the results using Reciprocal Rank Fusion (RRF)

# hybrid retrieval function containing both semantic search and keyword search
# candidate_n: number of candidate chunks selected by each retrieval method
# top_n: after combining the two rankings, the top n chunks are passed to the reranker (reranker is the next paragraph)
# (basically get 20 candidates per method, combine them and keep top 10, next paragraph: rerank them and keep top 5 to give the LLM)
def hybrid_retrieve(query, corpus, faiss_index, bm25_index, top_n=10, candidate_n=20, rrf_n=60):

    
    # SEMANTIC SEARCH:
    # embed the query
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    
    # normalize the query vector to length 1 (same thing that we did with chunks embeddings)
    query_embedding = query_embedding.reshape(1, -1) 
    faiss.normalize_L2(query_embedding)
    
    # search the candidate_n chunks most similar to the query in terms of embedding/cosine similarity/meaning
    # returns scores (cosine similarities) and indices (positions of the chunkin the corpus list)
    scores, indices = faiss_index.search(query_embedding, candidate_n)
    semantic_results = indices[0].tolist()

    
    # BM25 SEARCH:
    # tokenize the query
    tokenized_query = query.lower().split()
    
    # get the candidate_n chunks that contain query keywords the most
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_results = np.argsort(bm25_scores)[::-1][:candidate_n].tolist()

    
    # RRF:
    # combine the two rankings using RRF
    # basically each candidate chunk gets a score based on how high it appears in the faiss ranking 
    # and a score based on how high it appears in bm25 ranking
    # then these two scores are combined and all the chunks are reranked (this is not reranking though) based on the total score
    # finally pick only the top n chunks of that final ranking
    # mathematically the score of each chunk is:
    # - faiss score = 1/(60 + position faiss)
    # - bm35 score = 1/(60 + position bm25)
    # - total rrf score = faiss score + bm25 score

    rrf_scores = {}
    
    # assign RRF score from semantic ranking
    for rank, idx in enumerate(semantic_results):
        if idx not in rrf_scores:
            rrf_scores[idx] = 0.0
        rrf_scores[idx] += 1 / (rrf_n + rank + 1)
    
    # add RRF score from BM25 ranking 
    for rank, idx in enumerate(bm25_results):
        if idx not in rrf_scores:
            rrf_scores[idx] = 0.0
        rrf_scores[idx] += 1 / (rrf_n + rank + 1)
    
    # sort all chunks by their total RRF score descending and take the top n
    top_n_indices = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_n]
    
    # retrieve the actual text of the top n chunks using their indices
    top_n_chunks = [corpus[idx] for idx in top_n_indices]

    
    # return the text of the top n chunks and their indices
    return top_n_chunks, top_n_indices

# 4. RERANKING

In [ ]:
# RERANKING
# reranking obtains n chunks from the retrieval and reranks them more carefully and precisely
# cross-encoder is more precise than retrieval because it reads query and chunk together (while retrieval is bi-encoder)
# then we will keep only top k chunks that will be injected in the LLM prompt

# reranker model is cross-encoder/ms-marco-MiniLM-L-6-v2
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# reranker function takes the query and the top n chunks from hybrid retrieval
# feeds each (query, chunk) pair to the cross-encoder and gets a relevance score
# returns chunks sorted by reranker score descending
# after reranking only top k chunks will be kept and injected into the LLM prompt
def rerank(query, chunks, top_k=5):
    
    # build list of (query, chunk) pairs
    # same query is paired with each candidate chunk separately
    pairs = [(query, chunk) for chunk in chunks]
    
    # run the cross-encoder on all pairs and returns a score for each
    # then order the chunks in descending order based on the score
    scores = reranker.predict(pairs)
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    
    # select the top_k chunks
    top_chunks = [chunk for chunk, score in ranked[:top_k]]

    # return the text of the top k chunks
    return top_chunks

# 5. RAG PIPELINE

## LLM SETUP

In [ ]:
# LLM SETUP
# we use Llama 3.3 70billion parameters via the Groq API

# Groq API
groq_api_key = UserSecretsClient().get_secret("GROQAPI")

# ChatGroq is the LangChain wrapper for the Groq API
# the wrapper handles authentication and API calls to the model
# temperature=0: we want deterministic answers, no creativity needed for financial questions, accuracy is everything
llm_rag = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api_key, temperature=0)

## PROMPT TEMPLATE

In [ ]:
# PROMPT TEMPLATE
# defines how we present the prompt and the retrieved information to the LLM

# define the prompt
# system tells the model what it has to do
# context will be replaced with the retrieved documents
# user will be replaced with the user question
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst assistant.
    Answer the user's question using only the context provided below.
    If the answer is not contained in the context, say "I don't have enough information to answer this question."
    Be precise with numbers, dates, and financial figures.
    
    Context: {context}"""),
    
    ("user", "{question}")
])

# the | operator connects prompt template and llm
chain = prompt_template | llm_rag

## RAG PIPELINE

In [ ]:
# RAG PIPELINE
# the following function orchestrates retrieval, reranking, building context, calling LLM

# rag function gets in input the user question and the parameters needed by the other functions previously defined
# then it calls all those other functions in order to inject the LLM prompt with the retrieved context
# finally it receives the LLM answer to the question 
def rag(question, corpus, faiss_index, bm25_index, candidate_n=20, top_n=10, top_k=5):

    
    # call hybrid retrieval function
    retrieved_chunks, retrieved_indices = hybrid_retrieve(question, corpus, faiss_index, bm25_index, top_n=top_n, candidate_n=candidate_n)
    
    # call rerank function
    reranked_chunks = rerank(question, retrieved_chunks, top_k=top_k)
    
    # join all chunks into one single string that will be passed as context to the LLM
    context = "\n\n".join(reranked_chunks)
    
    # invoke the LLM with context and question and get its response
    response = chain.invoke({"context": context, "question": question})
    
    # return the response
    return response.content

# 6. SUMMARIZER

In [ ]:
# SUMMARIZER (map-reduce)
# it makes sense to have a first overview/summary of the 10-K right when it is uploaded, so we are building this functionality too
# we build the summarizer this way: 
# we only consider the first 100 chunks that contain the business overview, otherwise we immediately run out of tokens for the model
# we then group chunks in groups of 10 this way we will make even less API calls (soo 10 groups of 10 chunks)
# then we ask a LLM to summarize each group of 10 chunks giving us a summary for each group
# finally we give the LLM all these summaries and ask the LLM to make a summary of the summaries

def summarize_document(corpus, group_size=10):
    
    # split corpus into groups of group_size=10 consecutive chunks up to chunk 100
    groups = [corpus[i:i+group_size] for i in range(0, min(100, len(corpus)), group_size)]

    # prompt for the map phase: summarize each block concisely
    # we explicitly tell the model to ignore boilerplate and focus on substance
    # max 150 words keeps each mini-summary short so the reduce phase does not overflow
    map_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a financial analyst reading a section of a 10-K annual report.
        Summarize the key information in the text below in maximum 150 words.
        Focus on: business description, products and services, strategy, revenue drivers, and risk factors.
        Ignore: legal boilerplate, administrative details, and stock market information.
        If the section contains no relevant business or financial information, respond with exactly: NO RELEVANT CONTENT"""),
        ("human", "{text}")
    ])
    map_chain = map_prompt | llm_rag

    # summarize each group independently and collect mini-summaries
    mini_summaries = []
    for i, group in enumerate(groups):
        # join the 10 chunks of this group into one text block
        group_text = "\n\n".join(group)
        # call the LLM and get the mini-summary for this group
        response = map_chain.invoke({"text": group_text})
        mini_summary = response.content
        # skip groups that contained only boilerplate
        if "NO RELEVANT CONTENT" not in mini_summary:
            mini_summaries.append(mini_summary)

    # concatenate all mini-summaries into one text
    all_mini_summaries = "\n\n".join(mini_summaries)

    # prompt the LLM to make a summary of the summaries
    reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior financial analyst writing an executive summary of a SEC 10-K annual report.
    Based on the section summaries below, write a structured report with exactly these sections:

    COMPANY NAME
    
    **Business Overview**
    What the company does, its main products and services, key markets and business segments.

    **Revenue Drivers and Business Segments**
    Key sources of revenue, main business segments and what drives growth in each.
    
    **Key Risk Factors**
    The most material operational and financial risks. Ignore generic legal risks and boilerplate.
    
    **Strategic Outlook**
    Growth initiatives, investments, and management priorities for the future.

    Rules:
    - Start with the full company name and stock ticker on the first line, before any section.
    - Each section must be 3-5 sentences
    - Be precise with numbers and dates
    - Ignore legal boilerplate, footnotes, and administrative details"""),
    ("human", "{text}")
    ])
    reduce_chain = reduce_prompt | llm_rag
        
    # call the LLM and get the summary of the summaries
    final_summary = reduce_chain.invoke({"text": all_mini_summaries})

    return final_summary.content

# 7. GRADIO INTERFACE

## PROCESS PDF

In [ ]:
# PROCESS PDF FUNCTION

def process_pdf(file):

    # extract the path of the uploaded PDF
    pdf_path = file.name

    # call the chunking function and save the chunks in the variable chunks
    chunks = chunking(pdf_path)

    # call embedding and indexing function
    corpus, faiss_index, bm25_index = embedding_indexing(chunks)

    # call summarizer function and return the summary
    summary = summarize_document(corpus)
    return summary, corpus, faiss_index, bm25_index

## GRADIO UI

In [ ]:
# GRADIO UI

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    # gr.State() creates an isolated memory for every user
    # each user gets their own corpus, faiss_index and bm25_index based on their PDF
    # so multiple users uploading different PDFs don't overwrite each other's PDF
    corpus_state = gr.State()
    faiss_state = gr.State()
    bm25_state = gr.State()
    
    gr.Markdown("# 10-K FINANCIAL ANALYST")
    gr.Markdown("Upload a company's 10-K filing and ask questions about it!")

    with gr.Row():
        # user inputs
        with gr.Column():
            # text box
            in_text = gr.Textbox(label="Ask a question", placeholder="e.g. What were total revenues in 2024?")
            # upload pdf
            pdf_upload = gr.File(label="Upload PDF", file_types=[".pdf"])
            with gr.Row():
                # run button
                run_btn = gr.Button("Ask", variant="primary")
        # rag outputs
        with gr.Column():
            # rag reply
            out_text = gr.Textbox(label="Answer", lines=6)
            # summary
            out_summary = gr.Textbox(label="Document Summary", lines=12)

    # when PDF is uploaded it is used as input for process_pdf function
    # it does chunking, embedding, indexing and returns the summary
    pdf_upload.change(
        fn=process_pdf, 
        inputs=pdf_upload, 
        outputs=[out_summary, corpus_state, faiss_state, bm25_state]
    )
    # when execute button is clicked the rag function is called with input in_text and output out_text
    run_btn.click(
        fn=rag, 
        inputs=[in_text, corpus_state, faiss_state, bm25_state], 
        outputs=[out_text]
    )

# no demo.launch() because it is launched by FastAPI

# 8. RETRIEVAL EVALUATION (MRR)

In [ ]:
# MRR EVALUATION
# we evaluate the quality of the hybrid retrieval using MRR
# first after reading the PDF (in this case it was Tesla 10-K) we thought of 10 questions that simulate what a user might ask the RAG
# then we found the correct answer in the pdf and this allowed us to build a dict with the questions and the keywords that represent the answer
# at this point we ask the questions to the RAG and it will rank the chunks based on how relevant it thinks they are to the question
# but since we already know the answer we can easily find the chunk with the correct answer/keywords and see where it is placed in the ranking
# we want that chunk to be as close to the top as possible to consider the RAG precise

# the 10 questions
eval_dataset = [
    {"question": "What were Tesla's total revenues in 2024?",
     "keywords": ["97,690", "total revenues"]},
    {"question": "What was Tesla's net income in 2024?",
     "keywords": ["7,153", "net income"]},
    {"question": "What was Tesla's gross profit in 2024?",
     "keywords": ["17,450", "gross profit"]},
    {"question": "How much did Tesla spend on research and development in 2024?",
     "keywords": ["4,540", "research and development"]},
    {"question": "What was Tesla's total inventory as of December 31 2024?",
     "keywords": ["12,017", "total"]},
    {"question": "What was Tesla's total property plant and equipment net in 2024?",
     "keywords": ["35,836", "property, plant and equipment, net"]},
    {"question": "What was Tesla's total debt and finance leases as of December 31 2024?",
     "keywords": ["5,757", "total debt and finance leases"]},
    {"question": "What was Tesla's provision for income taxes in 2024?",
     "keywords": ["1,837", "provision for"]},
    {"question": "What were Tesla's total accrued liabilities in 2024?",
     "keywords": ["10,723", "total"]},
    {"question": "What was Tesla's income before income taxes in 2024?",
     "keywords": ["8,990", "income before income taxes"]},
]

In [ ]:
# create corpus, faiss_index and bm25_index globally so compute_mrr can access them
# normally they only exist inside the Gradio pipeline, passed between functions and never exposed globally
pdf_path = "/kaggle/input/datasets/filippotenani/annualreport/tsla10k.pdf"
chunks = chunking(pdf_path)
# this makes them global so compute_mrr can see them
corpus, faiss_index, bm25_index = embedding_indexing(chunks)

In [ ]:
# MRR CALCULATION
# MRR = average of 1/rank_found across all questions

# run each question and check at what rank the correct chunk appears after reranking
# this measures the quality of the retrieval pipeline
def compute_mrr(eval_dataset, corpus, faiss_index, bm25_index, candidate_n=20, top_n=10, top_k=5):

    reciprocal_ranks = []

    for item in eval_dataset:

        # retrieve the top n chunks for the current question
        retrieved_chunks, _ = hybrid_retrieve(item["question"], corpus, faiss_index, bm25_index, top_n=top_n, candidate_n=candidate_n)

        # rerank the retrieved chunks using the cross-encoder
        reranked_chunks = rerank(item["question"], retrieved_chunks, top_k=top_k)

        # scan the reranked chunks in order and find the first one that contains all keywords
        # rank+1 because we want first place to be 1 not 0
        rank_found = None
        for rank, chunk in enumerate(reranked_chunks):
            if all(kw.lower() in chunk.lower() for kw in item["keywords"]):
                rank_found = rank + 1
                break

        # if the correct chunk was not found, its contribution to MRR is 0
        # otherwise the contribution is 1/rank_found
        # the higher the mrr (the closest to 1 because it is 0-1)the lower the rank the better the retrieval
        if rank_found is None:
            reciprocal_ranks.append(0.0)
            print("NOT FOUND", item["question"])
        else:
            reciprocal_ranks.append(1 / rank_found)
            print("Rank", rank_found, "RR", round(1 / rank_found, 2), item["question"])

    # average all reciprocal ranks to get the final MRR score
    mrr = round(np.mean(reciprocal_ranks), 4)
    print("\nMRR: " + str(mrr))

compute_mrr(eval_dataset, corpus, faiss_index, bm25_index)

In [ ]:
# RESULTS:
# Rank 1 RR 1.0 What were Tesla's total revenues in 2024?
# Rank 1 RR 1.0 What was Tesla's net income in 2024?
# Rank 2 RR 0.5 What was Tesla's gross profit in 2024?
# Rank 1 RR 1.0 How much did Tesla spend on research and development in 2024?
# Rank 1 RR 1.0 What was Tesla's total inventory as of December 31 2024?
# Rank 1 RR 1.0 What was Tesla's total property plant and equipment net in 2024?
# NOT FOUND What was Tesla's total debt and finance leases as of December 31 2024?
# Rank 1 RR 1.0 What was Tesla's provision for income taxes in 2024?
# Rank 2 RR 0.5 What were Tesla's total accrued liabilities in 2024?
# Rank 2 RR 0.5 What was Tesla's income before income taxes in 2024?

# MRR: 0.75

# 9. FASTAPI

In [ ]:
# FASTAPI (MAIN)
# this is the entry point of the application
# it creates the FastAPI server, adds a health check endpoint,
# and mounts the Gradio UI so everything runs on a single server

import uvicorn
from fastapi import FastAPI
import gradio as gr
from gradioui import demo

# create the FastAPI application
app = FastAPI()

# health check endpoint
# returns a simple ok message to confirm the server is running
# Docker and HuggingFace Spaces use this to verify the container is alive
@app.get("/health")
def health():
    return {"status": "ok"}

# mount the Gradio app inside FastAPI at the root route
# gr.mount_gradio_app() takes the FastAPI app, the Gradio demo, and the path
app = gr.mount_gradio_app(app, demo, path="/")

# start the server with uvicorn when this file is run directly
# host 0.0.0.0 means the server accepts connections from outside the container (required for Docker)
# port 7860 is the standard port for HuggingFace Spaces
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=7860)

# 10. DOCKER

In [ ]:
# DOCKER
# builds a Docker container that runs the RAG application
# the container packages Python, all dependencies, and the app code together
# so it runs identically on everybody's computer

# start the container from the official Python 3.11 slim image
FROM python:3.11-slim

# set the working directory inside the container
WORKDIR /app

# copy requirements.txt inside the container
COPY requirements.txt .

# install all Python dependencies from requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# copy all project files into the container
COPY . .

# create an unprivileged user for safety reasons
# this way who uses the app will use it as a user not admin
RUN useradd -m -u 1000 user
USER user

# tell Docker that the container listens on port 7860
# this is the standard port for HuggingFace Spaces
EXPOSE 7860

# start the application when the container runs
CMD ["python", "main.py"]